## IMPORTS

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import sys
import duckdb
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings
warnings.filterwarnings('ignore')

# Import shared utilities
sys.path.insert(0, os.path.abspath('.'))
from notebook_utils import (
    drop_high_missing_columns, impute_features,
    plot_trajectory_distribution, plot_boxplots_with_stats,
    plot_roc_pr_curves, print_statistical_comparisons, train_repeated_cv, biomarker_summary_stats
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✓ Imports successful")
print("✓ Shared utilities loaded")

✓ Imports successful
✓ Shared utilities loaded


## LOAD DATA

In [2]:
# Connect to HiRiD DuckDB
db_path = '/home/gaga/data/physionet/HiRiD/hirid.duckdb'
conn = duckdb.connect(db_path, read_only=True)

print(f"✓ Connected to HiRiD database: {db_path}")

✓ Connected to HiRiD database: /home/gaga/data/physionet/HiRiD/hirid.duckdb


In [3]:
# Load patient cohort - sepsis patients
# Define sepsis as: infection + organ dysfunction (SOFA-like)
# Infection: elevated CRP or procalcitonin or WBC

n_sample_patients = 20000

# Step 1: Get patient IDs with signs of infection
infection_query = f"""
SELECT DISTINCT CAST(o.patientid AS INTEGER) as patientid
FROM observations o
WHERE 
    (
        -- CRP > 100 mg/L
        (o.variableid = '20002200' AND CAST(o.value AS DOUBLE) > 100)
        -- Procalcitonin > 0.5 ug/L
        OR (o.variableid = '24000570' AND CAST(o.value AS DOUBLE) > 0.5)
        -- WBC < 4 or > 12 G/L
        OR (o.variableid = '20000700' AND (CAST(o.value AS DOUBLE) < 4 OR CAST(o.value AS DOUBLE) > 12))
        -- Temperature < 36 or > 38.3
        OR (o.variableid = '410' AND (CAST(o.value AS DOUBLE) < 36 OR CAST(o.value AS DOUBLE) > 38.3))
    )
LIMIT {n_sample_patients}
"""

infection_patients = conn.execute(infection_query).fetchdf()['patientid'].tolist()
print(f"Step 1: Found {len(infection_patients):,} patients with infection signs")

# Step 2: Get patient demographics for those with infection
patient_query = f"""
SELECT 
    CAST(g.patientid AS INTEGER) as patientid,
    CAST(g.admissiontime AS TIMESTAMP) as admission_time,
    g.sex,
    CAST(g.age AS INTEGER) as age,
    g.discharge_status,
    COUNT(DISTINCT o.datetime) as n_observations,
    EPOCH(MAX(CAST(o.datetime AS TIMESTAMP)) - MIN(CAST(o.datetime AS TIMESTAMP))) / 86400.0 as los_days
FROM ref_general_table g
INNER JOIN observations o ON g.patientid = o.patientid
WHERE CAST(g.patientid AS INTEGER) IN {tuple(infection_patients)}
GROUP BY g.patientid, g.admissiontime, g.sex, g.age, g.discharge_status
HAVING 
    EPOCH(MAX(CAST(o.datetime AS TIMESTAMP)) - MIN(CAST(o.datetime AS TIMESTAMP))) / 86400.0 >= 2.0
    AND COUNT(DISTINCT o.datetime) >= 100
"""

patient_df = conn.execute(patient_query).fetchdf()

print(f"\n✓ Loaded {len(patient_df):,} sepsis patients")
print(f"  Mean LOS: {patient_df['los_days'].mean():.1f} days")
print(f"  Mean age: {patient_df['age'].mean():.1f} years")
print(f"  Sex distribution: {patient_df['sex'].value_counts().to_dict()}")

Step 1: Found 19,631 patients with infection signs

✓ Loaded 7,497 sepsis patients
  Mean LOS: 6.5 days
  Mean age: 62.5 years
  Sex distribution: {'M': 4927, 'F': 2570}


In [4]:
# Sample subset for analysis
n_patients = min(5000, len(patient_df))
patient_subset = patient_df.sample(n_patients, random_state=920)['patientid'].tolist()

print(f"✓ Using {len(patient_subset):,} patients for analysis")

✓ Using 5,000 patients for analysis


In [5]:
# Load lactate measurements (primary biomarker for shock)
# Lactate IDs: 24000524 (arterial), 24000732, 24000485 (venous)
lactate_query = f"""
SELECT 
    CAST(o.patientid AS INTEGER) as patientid,
    CAST(o.datetime AS TIMESTAMP) as charttime,
    CAST(o.value AS DOUBLE) as lactate,
    CAST(g.admissiontime AS TIMESTAMP) as admission_time
FROM observations o
INNER JOIN ref_general_table g ON o.patientid = g.patientid
WHERE 
    o.variableid IN ('24000524', '24000732', '24000485')
    AND o.value IS NOT NULL
    AND CAST(o.value AS DOUBLE) > 0
    AND CAST(o.value AS DOUBLE) < 30  -- Remove outliers
    AND CAST(o.patientid AS INTEGER) IN {tuple(patient_subset)}
ORDER BY o.patientid, o.datetime
"""

print("Loading lactate measurements...")
lactate_df = conn.execute(lactate_query).fetchdf()

print(f"\n✓ Loaded {len(lactate_df):,} lactate measurements")
print(f"  Patients with lactate: {lactate_df['patientid'].nunique():,}")
if len(lactate_df) > 0:
    print(f"  Mean lactate: {lactate_df['lactate'].mean():.2f} mmol/L")
    print(f"  Median lactate: {lactate_df['lactate'].median():.2f} mmol/L")

Loading lactate measurements...

✓ Loaded 127,459 lactate measurements
  Patients with lactate: 4,858
  Mean lactate: 1.94 mmol/L
  Median lactate: 1.30 mmol/L


In [6]:
# Load platelet and WBC measurements (additional biomarkers)
# Platelets ID: 20000110 (G/L)
# Leukocytes (WBC) ID: 20000700 (G/L)
platelet_query = f"""
SELECT 
    CAST(o.patientid AS INTEGER) as patientid,
    CAST(o.datetime AS TIMESTAMP) as charttime,
    CAST(o.value AS DOUBLE) as platelets,
    CAST(g.admissiontime AS TIMESTAMP) as admission_time
FROM observations o
INNER JOIN ref_general_table g ON o.patientid = g.patientid
WHERE 
    o.variableid = '20000110'
    AND o.value IS NOT NULL
    AND CAST(o.value AS DOUBLE) > 0
    AND CAST(o.value AS DOUBLE) < 1500  -- Remove outliers (G/L)
    AND CAST(o.patientid AS INTEGER) IN {tuple(patient_subset)}
ORDER BY o.patientid, o.datetime
"""

wbc_query = f"""
SELECT 
    CAST(o.patientid AS INTEGER) as patientid,
    CAST(o.datetime AS TIMESTAMP) as charttime,
    CAST(o.value AS DOUBLE) as wbc,
    CAST(g.admissiontime AS TIMESTAMP) as admission_time
FROM observations o
INNER JOIN ref_general_table g ON o.patientid = g.patientid
WHERE 
    o.variableid = '20000700'
    AND o.value IS NOT NULL
    AND CAST(o.value AS DOUBLE) > 0.1
    AND CAST(o.value AS DOUBLE) < 100  -- Remove outliers (G/L)
    AND CAST(o.patientid AS INTEGER) IN {tuple(patient_subset)}
ORDER BY o.patientid, o.datetime
"""

print("Loading platelet and WBC measurements...")
platelet_df = conn.execute(platelet_query).fetchdf()
wbc_df = conn.execute(wbc_query).fetchdf()

print(f"\n✓ Loaded {len(platelet_df):,} platelet measurements")
print(f"  Patients with platelets: {platelet_df['patientid'].nunique():,}")
print(f"✓ Loaded {len(wbc_df):,} WBC measurements")
print(f"  Patients with WBC: {wbc_df['patientid'].nunique():,}")
if len(platelet_df) > 0:
    print(f"  Median platelets: {platelet_df['platelets'].median():.1f} G/L")
if len(wbc_df) > 0:
    print(f"  Median WBC: {wbc_df['wbc'].median():.1f} G/L")

Loading platelet and WBC measurements...

✓ Loaded 89,163 platelet measurements
  Patients with platelets: 4,993
✓ Loaded 90,629 WBC measurements
  Patients with WBC: 4,984
  Median platelets: 167.0 G/L
  Median WBC: 11.4 G/L


In [7]:
# Load vasopressor administration (pharma table)
vasopressor_query = f"""
SELECT 
    CAST(p.patientid AS INTEGER) as patientid,
    CAST(p.givenat AS TIMESTAMP) as charttime,
    p.pharmaid,
    CAST(g.admissiontime AS TIMESTAMP) as admission_time
FROM pharma_records p
INNER JOIN ref_general_table g ON p.patientid = g.patientid
WHERE 
    -- Noradrenalin
    p.pharmaid IN ('1000462', '1000656', '1000657', '1000658')
    -- Adrenalin
    OR p.pharmaid IN ('71', '1000750', '1000649', '1000650', '1000655')
    -- Vasopressin
    OR p.pharmaid IN ('112', '113')
    -- Dobutamine
    OR p.pharmaid = '426'
    AND CAST(p.patientid AS INTEGER) IN {tuple(patient_subset)}
ORDER BY p.patientid, p.givenat
"""

print("Loading vasopressor data...")
vasopressor_df = conn.execute(vasopressor_query).fetchdf()

print(f"\n✓ Loaded {len(vasopressor_df):,} vasopressor records")
print(f"  Patients on vasopressors: {vasopressor_df['patientid'].nunique():,}")

Loading vasopressor data...

✓ Loaded 1,734,139 vasopressor records
  Patients on vasopressors: 12,258


## PREPROCESS

In [8]:
# Calculate baseline lactate (first 24h)
lactate_df['charttime'] = pd.to_datetime(lactate_df['charttime'])
lactate_df['admission_time'] = pd.to_datetime(lactate_df['admission_time'])

first_24h = lactate_df[lactate_df['charttime'] <= lactate_df['admission_time'] + pd.Timedelta(hours=24)]

baseline_lactate = first_24h.groupby('patientid')['lactate'].first().reset_index()
baseline_lactate.columns = ['patientid', 'baseline_lactate']

print(f"\nBaseline Lactate:")
print(f"   Patients with baseline: {len(baseline_lactate):,}")
print(f"   Mean: {baseline_lactate['baseline_lactate'].mean():.2f} mmol/L")
print(f"   Median: {baseline_lactate['baseline_lactate'].median():.2f} mmol/L")


Baseline Lactate:
   Patients with baseline: 4,704
   Mean: 2.60 mmol/L
   Median: 1.70 mmol/L


In [9]:
# Calculate baselines for platelets and WBC (first 24h)
platelet_df['charttime'] = pd.to_datetime(platelet_df['charttime'])
platelet_df['admission_time'] = pd.to_datetime(platelet_df['admission_time'])
wbc_df['charttime'] = pd.to_datetime(wbc_df['charttime'])
wbc_df['admission_time'] = pd.to_datetime(wbc_df['admission_time'])

platelet_first_24h = platelet_df[platelet_df['charttime'] <= platelet_df['admission_time'] + pd.Timedelta(hours=24)]
wbc_first_24h = wbc_df[wbc_df['charttime'] <= wbc_df['admission_time'] + pd.Timedelta(hours=24)]

baseline_platelets = platelet_first_24h.groupby('patientid')['platelets'].first().reset_index()
baseline_platelets.columns = ['patientid', 'baseline_platelets']

baseline_wbc = wbc_first_24h.groupby('patientid')['wbc'].first().reset_index()
baseline_wbc.columns = ['patientid', 'baseline_wbc']

print("\nBaseline Platelets / WBC:")
print(f"   Patients with platelet baseline: {len(baseline_platelets):,}")
print(f"   Median platelets: {baseline_platelets['baseline_platelets'].median():.1f} G/L")
print(f"   Patients with WBC baseline: {len(baseline_wbc):,}")
print(f"   Median WBC: {baseline_wbc['baseline_wbc'].median():.1f} G/L")


Baseline Platelets / WBC:
   Patients with platelet baseline: 4,794
   Median platelets: 172.0 G/L
   Patients with WBC baseline: 4,784
   Median WBC: 11.3 G/L


In [10]:
# Filter patients with sufficient measurements
lactate_counts = lactate_df.groupby('patientid').size()
valid_patients = lactate_counts[lactate_counts >= 3].index

lactate_filtered = lactate_df[lactate_df['patientid'].isin(valid_patients)]
patient_final = patient_df[patient_df['patientid'].isin(valid_patients)].merge(
    baseline_lactate, on='patientid', how='inner'
).merge(
    baseline_platelets, on='patientid', how='left'
).merge(
    baseline_wbc, on='patientid', how='left'
)

print(f"\nFiltering:")
print(f"   Patients with ≥3 lactate measurements: {len(patient_final):,}")


Filtering:
   Patients with ≥3 lactate measurements: 4,640


In [11]:
# Align platelets and WBC to valid sepsis patients and build daily values
valid_ids = set(patient_final['patientid'])

platelet_filtered = platelet_df[platelet_df['patientid'].isin(valid_ids)]
wbc_filtered = wbc_df[wbc_df['patientid'].isin(valid_ids)]

# Merge baselines
platelet_filtered = platelet_filtered.merge(baseline_platelets, on='patientid', how='left')
wbc_filtered = wbc_filtered.merge(baseline_wbc, on='patientid', how='left')

# Build time variables
platelet_filtered['time_days'] = (platelet_filtered['charttime'] - platelet_filtered['admission_time']).dt.total_seconds() / 86400
platelet_filtered['time_day'] = platelet_filtered['time_days'].astype(int)

wbc_filtered['time_days'] = (wbc_filtered['charttime'] - wbc_filtered['admission_time']).dt.total_seconds() / 86400
wbc_filtered['time_day'] = wbc_filtered['time_days'].astype(int)


print("\n✓ Platelet/WBC daily values prepared")


✓ Platelet/WBC daily values prepared


In [12]:
# Create time series
lactate_ts = lactate_filtered.merge(
    patient_final[['patientid', 'baseline_lactate', 'admission_time']], 
    on=['patientid', 'admission_time'], 
    how='left'
)

lactate_ts['time_days'] = (lactate_ts['charttime'] - lactate_ts['admission_time']).dt.total_seconds() / 86400
lactate_ts['time_day'] = lactate_ts['time_days'].astype(int)

# Rename for trajectory script
lactate_ts = lactate_ts.rename(columns={'lactate': 'lab_value'})

print(f"\n✓ Time series created")
print(f"   Total measurements: {len(lactate_ts):,}")
print(f"   Time range: {lactate_ts['time_days'].min():.1f} to {lactate_ts['time_days'].max():.1f} days")


✓ Time series created
   Total measurements: 127,323
   Time range: -0.0 to 28.0 days


In [13]:
# Create platelets time series
platelet_ts = platelet_filtered.merge(
    patient_final[['patientid', 'baseline_platelets', 'admission_time']], 
    on=['patientid', 'admission_time'], 
    how='left'
)

platelet_ts['time_days'] = (platelet_ts['charttime'] - platelet_ts['admission_time']).dt.total_seconds() / 86400
platelet_ts['time_day'] = platelet_ts['time_days'].astype(int)

# Rename for trajectory script
platelet_ts = platelet_ts.rename(columns={'platelets': 'lab_value'})
print(f"\n✓ Time series created")
print(f"   Total measurements: {len(platelet_ts):,}")
print(f"   Time range: {platelet_ts['time_days'].min():.1f} to {platelet_ts['time_days'].max():.1f} days")

# Create wbc time series
wbc_ts = wbc_filtered.merge(
    patient_final[['patientid', 'baseline_wbc', 'admission_time']], 
    on=['patientid', 'admission_time'],
    how='left'
)
wbc_ts['time_days'] = (wbc_ts['charttime'] - wbc_ts['admission_time']).dt.total_seconds() / 86400
wbc_ts['time_day'] = wbc_ts['time_days'].astype(int)
# Rename for trajectory script
wbc_ts = wbc_ts.rename(columns={'wbc': 'lab_value'})
print(f"\n✓ Time series created")
print(f"   Total measurements: {len(wbc_ts):,}")
print(f"   Time range: {wbc_ts['time_days'].min():.1f} to {wbc_ts['time_days'].max():.1f} days")


✓ Time series created
   Total measurements: 85,366
   Time range: -0.0 to 28.0 days

✓ Time series created
   Total measurements: 86,687
   Time range: -0.0 to 28.0 days


In [14]:
lactate_ts = lactate_ts.drop_duplicates(subset=['patientid', 'charttime'])
platelet_ts = platelet_ts.drop_duplicates(subset=['patientid', 'charttime'])
wbc_ts = wbc_ts.drop_duplicates(subset=['patientid', 'charttime'])

In [15]:
# Save lactate time series for trajectory computation
output_path = '../../results/hirid/sepsis/lactate_timeseries.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

lactate_ts.to_csv(output_path, index=False)
platelet_ts.to_csv('../../results/hirid/sepsis/platelet_timeseries.csv', index=False)
wbc_ts.to_csv('../../results/hirid/sepsis/wbc_timeseries.csv', index=False)

print(f"\n✓ Saved lactate time series: {output_path}")
print(f"✓ Saved platelet time series: ../../results/hirid/sepsis/platelet_timeseries.csv")
print(f"✓ Saved WBC time series: ../../results/hirid/sepsis/wbc_timeseries.csv")
print(f"   Shape: {lactate_ts.shape}")


✓ Saved lactate time series: ../../results/hirid/sepsis/lactate_timeseries.csv
✓ Saved platelet time series: ../../results/hirid/sepsis/platelet_timeseries.csv
✓ Saved WBC time series: ../../results/hirid/sepsis/wbc_timeseries.csv
   Shape: (94603, 7)


## TRAJECTORY MODELING

Run standalone script for SLURM:
```bash
python ../hirid_sepsis_trajs.py --window-days 3.0 --n-batches 10
```

In [ ]:
# Load pre-computed trajectory probabilities
trajectory_probs_path = '../../results/hirid/sepsis/sepsis_trajectory_probs_bayes.csv'

if os.path.exists(trajectory_probs_path):
    trajectory_probs = pd.read_csv(trajectory_probs_path)
    print(f"✓ Loaded trajectory probabilities from: {trajectory_probs_path}")
    print(f"   Shape: {trajectory_probs.shape}")
else:
    print(f"⚠️ File not found: {trajectory_probs_path}")
    print("   Run: python ../hirid_sepsis_trajs.py")

In [ ]:
# Visualize trajectory distribution
plot_trajectory_distribution(trajectory_probs)

## SUMMARY FEATURES

## DEFINE OUTCOME

Septic shock defined as:
1. Hypotension (MAP < 65 mmHg) OR
2. Vasopressor requirement AND
3. Elevated lactate (> 2 mmol/L)

In [ ]:
# Configuration
PREDICTION_GAP_DAYS = 0.5  # 12-hour gap
PREDICTION_WINDOW_DAYS = 3.0  # Predict shock within 3 days

print(f"📋 Prediction Configuration:")
print(f"   Gap:    {PREDICTION_GAP_DAYS} days")
print(f"   Window: {PREDICTION_WINDOW_DAYS} days")
print(f"   Total:  {PREDICTION_GAP_DAYS + PREDICTION_WINDOW_DAYS} days lookahead")

# Prepare vasopressor timing
vasopressor_df['charttime'] = pd.to_datetime(vasopressor_df['charttime'])
vasopressor_df['admission_time'] = pd.to_datetime(vasopressor_df['admission_time'])
vasopressor_df['time_days'] = (vasopressor_df['charttime'] - vasopressor_df['admission_time']).dt.total_seconds() / 86400
vasopressor_df['time_day'] = vasopressor_df['time_days'].astype(int)

# Get patients on vasopressors by day
vasopressor_days = vasopressor_df.groupby(['patientid', 'time_day']).size().reset_index(name='vaso_count')
vasopressor_days['on_vasopressors'] = 1

print(f"\n✓ Vasopressor data prepared")
print(f"  Patient-days with vasopressors: {len(vasopressor_days):,}")

In [ ]:
# Define septic shock events (multimarker: lactate + platelets + WBC)
shock_events = []
excluded_counts = {'already_shock': 0, 'no_future_data': 0}

# Use trajectory_probs if loaded, otherwise use lactate_ts
if 'trajectory_probs' in locals():
    prediction_base = trajectory_probs.rename(columns={'lab_value': 'lactate'})
else:
    prediction_base = lactate_ts.rename(columns={'lab_value': 'lactate'})

# Enrich with daily platelets/WBC and baselines
prediction_base = prediction_base.merge(platelet_daily, on=['patientid', 'time_day'], how='left')
prediction_base = prediction_base.merge(wbc_daily, on=['patientid', 'time_day'], how='left')
prediction_base = prediction_base.merge(baseline_platelets, on='patientid', how='left')
prediction_base = prediction_base.merge(baseline_wbc, on='patientid', how='left')

for patientid, grp in prediction_base.groupby('patientid'):
    grp = grp.sort_values('time_day')
    baseline_lac = grp['baseline_lactate'].iloc[0] if 'baseline_lactate' in grp.columns else grp['lactate'].iloc[0]
    baseline_plt = grp['baseline_platelets'].iloc[0] if 'baseline_platelets' in grp.columns else np.nan
    baseline_wbc_val = grp['baseline_wbc'].iloc[0] if 'baseline_wbc' in grp.columns else np.nan
    
    for i in range(len(grp)):
        current_time = grp.iloc[i]['time_day']
        current_lactate = grp.iloc[i]['lactate']
        current_platelets = grp.iloc[i].get('platelets_daily', np.nan)
        current_wbc = grp.iloc[i].get('wbc_daily', np.nan)
        
        # Skip if already in shock (lactate > 4)
        if current_lactate > 4.0:
            excluded_counts['already_shock'] += 1
            continue
        
        # Define prediction window
        prediction_start = current_time + PREDICTION_GAP_DAYS
        prediction_end = current_time + PREDICTION_GAP_DAYS + PREDICTION_WINDOW_DAYS
        
        future_window = grp[
            (grp['time_day'] >= prediction_start) & 
            (grp['time_day'] <= prediction_end)
        ]
        
        if len(future_window) == 0:
            excluded_counts['no_future_data'] += 1
            continue
        
        # Check for shock: lactate > 2 AND vasopressor use
        high_lactate = (future_window['lactate'] > 2.0).any()
        
        # Check vasopressor use in window
        vaso_in_window = vasopressor_days[
            (vasopressor_days['patientid'] == patientid) &
            (vasopressor_days['time_day'] >= prediction_start) &
            (vasopressor_days['time_day'] <= prediction_end)
        ]
        
        on_vasopressor = len(vaso_in_window) > 0
        target_shock = int(high_lactate and on_vasopressor)
        
        platelet_fold_change = (
            current_platelets / baseline_plt if pd.notnull(current_platelets) and pd.notnull(baseline_plt) and baseline_plt > 0 else np.nan
        )
        platelet_low = int(pd.notnull(current_platelets) and current_platelets < 150)
        
        wbc_fold_change = (
            current_wbc / baseline_wbc_val if pd.notnull(current_wbc) and pd.notnull(baseline_wbc_val) and baseline_wbc_val > 0 else np.nan
        )
        wbc_high = int(pd.notnull(current_wbc) and current_wbc > 12)
        wbc_low = int(pd.notnull(current_wbc) and current_wbc < 4)
        wbc_abnormal = int(wbc_high or wbc_low)
        
        shock_events.append({
            'patientid': patientid,
            'time_day': current_time,
            'target_septic_shock': target_shock,
            'current_lactate': current_lactate,
            'baseline_lactate': baseline_lac,
            'lactate_fold_change': current_lactate / baseline_lac if baseline_lac > 0 else 1.0,
            'current_platelets': current_platelets,
            'baseline_platelets': baseline_plt,
            'platelet_fold_change': platelet_fold_change,
            'platelet_low': platelet_low,
            'current_wbc': current_wbc,
            'baseline_wbc': baseline_wbc_val,
            'wbc_fold_change': wbc_fold_change,
            'wbc_high': wbc_high,
            'wbc_low': wbc_low,
            'wbc_abnormal': wbc_abnormal,
            'prediction_gap_days': PREDICTION_GAP_DAYS,
            'prediction_window_days': PREDICTION_WINDOW_DAYS
        })

shock_outcomes_df = pd.DataFrame(shock_events)

print(f"\n📊 Outcome Definition Summary:")
print(f"   Total prediction windows: {len(shock_outcomes_df):,}")
print(f"   Unique patients: {shock_outcomes_df['patientid'].nunique():,}")
print(f"\n   Septic Shock Events:")
print(f"      Total events: {shock_outcomes_df['target_septic_shock'].sum():,}")
print(f"      Event rate: {100*shock_outcomes_df['target_septic_shock'].mean():.1f}%")
print(f"\n   Platelet data availability: {shock_outcomes_df['current_platelets'].notna().mean()*100:.1f}% of rows")
print(f"   WBC data availability: {shock_outcomes_df['current_wbc'].notna().mean()*100:.1f}% of rows")

## CONSTRUCT PREDICTION DATASET

In [ ]:
# Merge trajectory probabilities with outcomes
prediction_dataset = shock_outcomes_df.copy()

if 'trajectory_probs' in locals() and 'prob_stable' in trajectory_probs.columns:
    prediction_dataset = prediction_dataset.merge(
        trajectory_probs[[
            'patientid', 'time_day', 
            'prob_stable', 'prob_gradual_increase', 'prob_rapid_increase'
        ]],
        on=['patientid', 'time_day'],
        how='left'
    )
    
    prediction_dataset['prob_worsening'] = (
        prediction_dataset['prob_gradual_increase'] + 
        prediction_dataset['prob_rapid_increase']
    )

# Merge patient demographics
patient_static = patient_final[['patientid', 'age', 'sex']].drop_duplicates()
prediction_dataset = prediction_dataset.merge(
    patient_static,
    on='patientid',
    how='left'
)

# Add lactate-derived features
prediction_dataset['lactate_elevated'] = (prediction_dataset['current_lactate'] > 2.0).astype(int)
prediction_dataset['platelet_very_low'] = (
    (prediction_dataset['current_platelets'] < 100) & prediction_dataset['current_platelets'].notna()
).astype(int)
prediction_dataset['wbc_high_flag'] = prediction_dataset['wbc_high']
prediction_dataset['wbc_low_flag'] = prediction_dataset['wbc_low']
prediction_dataset['wbc_abnormal_flag'] = prediction_dataset['wbc_abnormal']

# Load vitals/labs for dynamic features (comprehensive set matching Liver approach)
print(f"\n📋 Loading vitals and labs for dynamic feature engineering...")

# Get all relevant clinical variables from HiRID reference
key_vars_query = """
SELECT DISTINCT
    CAST("ID" AS INTEGER) as variableid,
    "Variable Name" as variable_name,
    "Unit" as unit
FROM ref_hirid_variable_reference
WHERE 
    "Variable Name" = 'Heart rate'
    OR "Variable Name" = 'Core body temperature'
    OR "Variable Name" = 'Invasive systolic arterial pressure'
    OR "Variable Name" = 'Invasive diastolic arterial pressure'
    OR "Variable Name" = 'Invasive mean arterial pressure'
    OR "Variable Name" = 'Respiratory rate'
    OR "Variable Name" = 'Peripheral oxygen saturation'
    OR "Variable Name" = 'Glucose [Moles/volume] in Serum or Plasma'
    OR "Variable Name" = 'Sodium [Moles/volume] in Blood'
    OR "Variable Name" = 'Potassium [Moles/volume] in Blood'
    OR "Variable Name" = 'Chloride [Moles/volume] in Blood'
    OR "Variable Name" = 'Bicarbonate [Moles/volume] in Arterial blood'
    OR "Variable Name" = 'Calcium.ionized [Moles/volume] in Blood'
    OR "Variable Name" = 'Magnesium [Moles/volume] in Blood'
    OR "Variable Name" = 'Phosphate [Moles/volume] in Blood'
    OR "Variable Name" = 'Urea [Moles/volume] in Venous blood'
    OR "Variable Name" = 'Creatinine [Moles/volume] in Blood'
    OR "Variable Name" = 'Hemoglobin [Mass/volume] in Blood'
    OR "Variable Name" = 'Leukocytes [#/volume] in Blood'
    OR "Variable Name" = 'Platelets [#/volume] in Blood'
    OR "Variable Name" = 'INR in Blood by Coagulation assay'
    OR "Variable Name" = 'Albumin [Mass/volume] in Serum or Plasma'
    OR "Variable Name" = 'Lactate [Mass/volume] in Arterial blood'
    OR "Variable Name" = 'Body weight'
"""

key_vars = conn.execute(key_vars_query).fetchdf()
print(f"✓ Found {len(key_vars)} clinical variables")

# Create variable ID to column name mapping
var_id_to_col = {}
for _, row in key_vars.iterrows():
    var_name = row['variable_name']
    var_id = str(row['variableid'])
    
    if 'Invasive systolic' in var_name:
        col_name = 'sbp_invasive'
    elif 'Invasive diastolic' in var_name:
        col_name = 'dbp_invasive'
    elif 'Invasive mean arterial' in var_name:
        col_name = 'mbp_invasive'
    elif 'Heart rate' in var_name:
        col_name = 'heart_rate'
    elif 'temperature' in var_name.lower():
        col_name = 'temperature'
    elif 'Respiratory rate' in var_name:
        col_name = 'respiratory_rate'
    elif 'oxygen saturation' in var_name.lower():
        col_name = 'spo2'
    elif 'Glucose' in var_name:
        col_name = 'glucose'
    elif 'Sodium' in var_name:
        col_name = 'sodium'
    elif 'Potassium' in var_name:
        col_name = 'potassium'
    elif 'Chloride' in var_name:
        col_name = 'chloride'
    elif 'Bicarbonate' in var_name:
        col_name = 'bicarbonate'
    elif 'Calcium' in var_name:
        col_name = 'calcium'
    elif 'Magnesium' in var_name:
        col_name = 'magnesium'
    elif 'Phosphate' in var_name:
        col_name = 'phosphate'
    elif 'Urea' in var_name:
        col_name = 'bun'
    elif 'Creatinine' in var_name:
        col_name = 'creatinine'
    elif 'Hemoglobin' in var_name:
        col_name = 'hemoglobin'
    elif 'Leukocytes' in var_name:
        col_name = 'wbc'
    elif 'Platelets' in var_name:
        col_name = 'platelets'
    elif 'INR' in var_name:
        col_name = 'inr'
    elif 'Albumin' in var_name:
        col_name = 'albumin'
    elif 'Lactate' in var_name:
        col_name = 'lactate'
    elif 'weight' in var_name.lower():
        col_name = 'weight'
    else:
        col_name = var_name.lower().replace(' ', '_')[:30]
    
    var_id_to_col[var_id] = col_name

# Load observations
var_ids_str = "', '".join(var_id_to_col.keys())
vitals_labs_query = f"""
SELECT 
    CAST(o.patientid AS INTEGER) as patientid,
    CAST(o.datetime AS TIMESTAMP) as charttime,
    o.variableid,
    CAST(o.value AS DOUBLE) as value,
    CAST(g.admissiontime AS TIMESTAMP) as admission_time
FROM observations o
INNER JOIN ref_general_table g ON o.patientid = g.patientid
WHERE 
    o.variableid IN ('{var_ids_str}')
    AND o.value IS NOT NULL
    AND CAST(o.patientid AS INTEGER) IN {tuple(patient_subset)}
ORDER BY o.patientid, o.datetime
"""

vitals_labs_raw = conn.execute(vitals_labs_query).fetchdf()
print(f"✓ Loaded {len(vitals_labs_raw):,} vitals/labs observations")

# Aggregate vitals/labs by time window
vitals_labs_raw['charttime'] = pd.to_datetime(vitals_labs_raw['charttime'])
vitals_labs_raw['admission_time'] = pd.to_datetime(vitals_labs_raw['admission_time'])
vitals_labs_raw['time_day'] = ((vitals_labs_raw['charttime'] - vitals_labs_raw['admission_time']).dt.total_seconds() / 86400).astype(int)
vitals_labs_raw['variable'] = vitals_labs_raw['variableid'].astype(str).map(var_id_to_col)

# Convert creatinine from µmol/L to mg/dL
if 'creatinine' in vitals_labs_raw['variable'].values:
    vitals_labs_raw.loc[vitals_labs_raw['variable'] == 'creatinine', 'value'] = \
        vitals_labs_raw.loc[vitals_labs_raw['variable'] == 'creatinine', 'value'] / 88.4

# Clean outliers with clinically plausible ranges
valid_ranges = {
    'heart_rate': (20, 250), 'temperature': (32, 42), 
    'sbp_invasive': (40, 250), 'dbp_invasive': (20, 180), 'mbp_invasive': (30, 200),
    'respiratory_rate': (4, 60), 'spo2': (50, 100),
    'glucose': (1, 50), 'sodium': (100, 160), 'potassium': (2, 10),
    'chloride': (80, 120), 'bicarbonate': (5, 50), 'calcium': (0.5, 3),
    'magnesium': (0.3, 5), 'phosphate': (0.3, 3), 'bun': (0, 150),
    'creatinine': (0.1, 15), 'hemoglobin': (3, 20), 'wbc': (0.5, 50),
    'platelets': (10, 1000), 'inr': (0.5, 15), 'albumin': (1, 6),
    'lactate': (0.1, 30), 'weight': (20, 300)
}

vitals_labs_clean = vitals_labs_raw.copy()
for var, (min_val, max_val) in valid_ranges.items():
    mask = vitals_labs_clean['variable'] == var
    if mask.any():
        vitals_labs_clean.loc[mask, 'value'] = vitals_labs_clean.loc[mask, 'value'].clip(min_val, max_val)

# Aggregate by time_day and variable
vitals_labs_agg = vitals_labs_clean.groupby(['patientid', 'time_day', 'variable'])['value'].agg(
    ['min', 'max', 'mean']
).reset_index()
vitals_labs_agg.columns = ['patientid', 'time_day', 'variable', 'min_val', 'max_val', 'mean_val']

# Pivot to wide format
vitals_labs_features_df = vitals_labs_agg.melt(
    id_vars=['patientid', 'time_day', 'variable'],
    value_vars=['min_val', 'max_val', 'mean_val'],
    var_name='stat',
    value_name='value'
)
vitals_labs_features_df['feature'] = \
    vitals_labs_features_df['variable'] + '_' + vitals_labs_features_df['stat'].str.replace('_val', '')
vitals_labs_features_df = vitals_labs_features_df.pivot_table(
    index=['patientid', 'time_day'],
    columns='feature',
    values='value'
).reset_index()

# Remove lactate/platelet/WBC features (already have as primary biomarkers)
vitals_labs_features_df = vitals_labs_features_df.drop(
    columns=[col for col in vitals_labs_features_df.columns 
             if col.startswith('lactate_') or col.startswith('platelets_') or col.startswith('wbc_')],
    errors='ignore'
)

print(f"✓ Aggregated {len(vitals_labs_features_df):,} vitals/labs timepoints")

# Merge vitals/labs
if len(vitals_labs_features_df) > 0:
    prediction_dataset = prediction_dataset.merge(
        vitals_labs_features_df,
        on=['patientid', 'time_day'],
        how='left'
    )
    print(f"✓ Merged {len(vitals_labs_features_df.columns) - 2} dynamic vitals/labs features")
    
    # Backfill weight only (more stable characteristic)
    if 'weight_mean' in prediction_dataset.columns:
        prediction_dataset['weight_mean'] = prediction_dataset.groupby('patientid')['weight_mean'].bfill()

print(f"\n📋 Final Prediction Dataset:")
print(f"   Rows: {len(prediction_dataset):,}")
print(f"   Columns: {len(prediction_dataset.columns)}")
print(f"   Unique patients: {prediction_dataset['patientid'].nunique():,}")

# Save
os.makedirs('../../results/hirid/sepsis', exist_ok=True)
prediction_dataset.to_csv('../../results/hirid/sepsis/sepsis_prediction_dataset.csv', index=False)
print(f"\n✓ Saved: results/hirid/sepsis/sepsis_prediction_dataset.csv")

## FIT MODELS

In [ ]:
# Define feature sets
feature_sets = {
    'Trajectory Only': [
        'prob_stable',
        'prob_gradual_increase',
        'prob_rapid_increase',
        'prob_worsening'
    ],
    
    'Static Only': [
        'lactate_fold_change',
        'lactate_elevated',
        'baseline_lactate',
        'platelet_fold_change',
        'platelet_low',
        'platelet_very_low',
        'baseline_platelets',
        'wbc_fold_change',
        'wbc_abnormal_flag',
        'wbc_high_flag',
        'wbc_low_flag',
        'baseline_wbc',
        'age',
    ],
    
    'Trajectory + Static': [
        'prob_stable',
        'prob_gradual_increase',
        'prob_rapid_increase',
        'prob_worsening',
        'lactate_fold_change',
        'lactate_elevated',
        'baseline_lactate',
        'platelet_fold_change',
        'platelet_low',
        'platelet_very_low',
        'baseline_platelets',
        'wbc_fold_change',
        'wbc_abnormal_flag',
        'wbc_high_flag',
        'wbc_low_flag',
        'baseline_wbc',
        'age',
    ],
}

print(f"📋 Feature Set Summary:")
for name, features in feature_sets.items():
    print(f"   {name:35s}: {len(features):2d} features")

In [ ]:
# Prepare data for modeling
prediction_clean = prediction_dataset.dropna(subset=feature_sets['Trajectory + Static'] + ['target_septic_shock'])

if 'sex' in prediction_clean.columns:
    prediction_clean['sex'] = prediction_clean['sex'].map({'m': 1, 'f': 0, 'M': 1, 'F': 0})

feature_cols = feature_sets['Trajectory + Static']
prediction_clean = impute_features(prediction_clean, feature_cols)

y = prediction_clean['target_septic_shock']
groups = prediction_clean['patientid']

print(f"\n📋 Final Dataset for Modeling:")
print(f"   Total samples: {len(y):,}")
print(f"   Positive class: {y.sum():,} ({100*y.mean():.1f}%)")
print(f"   Unique patients: {groups.nunique():,}")

## EVALUATE

In [ ]:
# Repeated Cross-validation
n_repeats = 10
n_folds = 5
n_folds_total = n_repeats * n_folds

comparison_results = {}

print(f"\n🎯 Training models with {n_repeats}-Repeat {n_folds}-Fold CV:\n")

for feature_set_name, feature_cols in feature_sets.items():
    print(f"{feature_set_name}")
    print("-" * 60)
    
    fold_metrics = train_repeated_cv(
        prediction_df=prediction_clean,
        feature_cols=feature_cols,
        target_col='target_septic_shock',
        group_col='patientid',
        n_repeats=n_repeats,
        n_folds=n_folds,
        random_state=920
    )
    
    comparison_results[feature_set_name] = fold_metrics
    
    print(f"  ROC-AUC:           {np.mean(fold_metrics['roc_auc']):.3f} ± {np.std(fold_metrics['roc_auc']):.3f}")
    print(f"  Average Precision: {np.mean(fold_metrics['avg_precision']):.3f} ± {np.std(fold_metrics['avg_precision']):.3f}\n")

In [ ]:
# Performance comparison
summary_df = pd.DataFrame([
    {
        'Model': name,
        'ROC-AUC': f"{np.mean(metrics['roc_auc']):.3f} ± {np.std(metrics['roc_auc']):.3f}",
        'Avg Precision': f"{np.mean(metrics['avg_precision']):.3f} ± {np.std(metrics['avg_precision']):.3f}"
    }
    for name, metrics in comparison_results.items()
])

print("\n📊 Performance Summary:")
print(summary_df.to_string(index=False))

# ROC & PR Curves
fig, (ax1, ax2) = plot_roc_pr_curves(
    comparison_results=comparison_results,
    outcome_df=prediction_clean,
    target_col='target_septic_shock',
    color_scheme='Set1'
)
plt.savefig('../../results/hirid/sepsis/model_comparison_curves.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: results/hirid/sepsis/model_comparison_curves.png")
plt.show()

In [ ]:
# Statistical significance testing
pairs_to_compare = [(0, 1), (1, 2)]

fig, (ax1, ax2) = plot_boxplots_with_stats(
    comparison_results=comparison_results,
    outcome_df=prediction_clean,
    target_col='target_septic_shock',
    pairs_to_compare=pairs_to_compare,
    n_folds_total=n_folds_total
)
plt.savefig('../../results/hirid/sepsis/model_comparison_boxplots.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: results/hirid/sepsis/model_comparison_boxplots.png")
plt.show()

print_statistical_comparisons(
    comparison_results=comparison_results,
    pairs_to_compare=pairs_to_compare
)

## SUMMARY STATISTICS

Generate lactate summary statistics (matching MIMIC's approach)

In [ ]:
# Generate lactate summary statistics (matching MIMIC's approach)
LOOKBACK_DAYS = 3  # Match trajectory window for Sepsis (3 days)

# Prefer lactate_ts if available, otherwise fallback to prediction_base or trajectory_probs
if 'lactate_ts' in locals():
    ts_df = lactate_ts.rename(columns={'lab_value': 'lactate'})
elif 'prediction_base' in locals():
    ts_df = prediction_base
elif 'trajectory_probs' in locals():
    ts_df = trajectory_probs.rename(columns={'lab_value': 'lactate'})
else:
    ts_df = None

if ts_df is not None and 'lactate' in ts_df.columns:
    lactate_summary_7d = biomarker_summary_stats(ts_df, 'lactate', LOOKBACK_DAYS)
    print(f"✓ Generated lactate summary stats: {len(lactate_summary_7d):,} timepoints")
    print(f"  Features: {list(lactate_summary_7d.columns)}")
    print(f"  Unique patients: {lactate_summary_7d['patientid'].nunique():,}")

    # Save summary stats
    os.makedirs('../../results/hirid/sepsis', exist_ok=True)
    lactate_summary_7d.to_csv('../../results/hirid/sepsis/lactate_summary_7d.csv', index=False)
    print(f"\n✓ Saved: results/hirid/sepsis/lactate_summary_7d.csv")
else:
    print("⚠️ Could not find lactate time series to compute summary stats.")

## EVALUATE (Expanded Feature Sets)

Run cross-validation using the expanded feature sets including dynamic vitals/labs and summary stats.

In [ ]:
# Redefine feature sets to include summary stats (filtered to available columns)
available_cols = set(prediction_dataset.columns)
summary_stat_cols = [c for c in (lactate_summary_7d.columns if 'lactate_summary_7d' in locals() else []) if c not in ['patientid','time_day']]
summary_stat_cols = [c for c in summary_stat_cols if c in available_cols]

base_sets = {
    'Trajectory Only': [
        'prob_stable','prob_gradual_increase','prob_rapid_increase','prob_worsening'
    ],
    'Static Only': [
        'lactate_fold_change','baseline_lactate','age','sex'
    ],
    'Static + Dynamic': [
        'lactate_fold_change','baseline_lactate','age','sex'
    ] + [c for c in prediction_dataset.columns if any(s in c for s in ['_min','_max','_mean'])]
}

feature_sets = {
    'Trajectory Only': [c for c in base_sets['Trajectory Only'] if c in available_cols],
    'Summary Stats Only': summary_stat_cols,
    'Trajectory + Summary Stats': [c for c in base_sets['Trajectory Only'] if c in available_cols] + summary_stat_cols,
    'Static Only': [c for c in base_sets['Static Only'] if c in available_cols],
    'Trajectory + Static': [c for c in (base_sets['Trajectory Only'] + base_sets['Static Only']) if c in available_cols],
    'Summary Stats + Static': summary_stat_cols + [c for c in base_sets['Static Only'] if c in available_cols],
    'Trajectory + Summary Stats + Static': [c for c in base_sets['Trajectory Only'] if c in available_cols] + summary_stat_cols + [c for c in base_sets['Static Only'] if c in available_cols],
    'Static + Dynamic': base_sets['Static + Dynamic'],
    'Trajectory + Static + Dynamic': [c for c in (base_sets['Trajectory Only'] + base_sets['Static Only']) if c in available_cols] + base_sets['Static + Dynamic'],
    'Summary Stats + Static + Dynamic': summary_stat_cols + base_sets['Static + Dynamic'],
    'Trajectory + Summary Stats + Static + Dynamic': [c for c in base_sets['Trajectory Only'] if c in available_cols] + summary_stat_cols + base_sets['Static + Dynamic'],
}

print(f"📋 Feature Set Summary:")
for name, features in feature_sets.items():
    print(f"   {name:50s}: {len(features):2d} features")

In [ ]:
# Improved imputation and dataset prep for modeling
prediction_clean = prediction_dataset.dropna(subset=['target_septic_shock']).copy()

# Encode gender
if 'sex' in prediction_clean.columns:
    prediction_clean['sex'] = prediction_clean['sex'].map({'M': 1, 'F': 0, 'm': 1, 'f': 0})

print(f"\n🔧 Applying intelligent imputation strategy:")

# Trajectory probabilities
traj_cols = ['prob_stable','prob_gradual_increase','prob_rapid_increase','prob_worsening']
for col in traj_cols:
    if col in prediction_clean.columns:
        prediction_clean[col] = prediction_clean.groupby('patientid')[col].fillna(method='ffill', limit=2)
        prediction_clean[col] = prediction_clean[col].fillna(0)
print(f"   ✓ Trajectory probs: ffill limit=2, then 0")

# Summary statistics
for col in summary_stat_cols:
    if col in prediction_clean.columns:
        prediction_clean[col] = prediction_clean.groupby('patientid')[col].fillna(method='ffill', limit=2)
print(f"   ✓ Summary stats: ffill limit=2")

# Vitals/labs
dyn_cols = [c for c in prediction_clean.columns if any(s in c for s in ['_min','_max','_mean'])]
for col in dyn_cols:
    prediction_clean[col] = prediction_clean.groupby('patientid')[col].fillna(method='ffill', limit=3)
print(f"   ✓ Vitals/labs: ffill limit=3")

# Current lactate
if 'current_lactate' in prediction_clean.columns:
    prediction_clean['current_lactate'] = prediction_clean.groupby('patientid')['current_lactate'].fillna(method='ffill', limit=2)

# Recompute derived features if present
if 'current_lactate' in prediction_clean.columns:
    prediction_clean['lactate_above_normal'] = (prediction_clean['current_lactate'] > 2.0).fillna(0).astype(int)

y = prediction_clean['target_septic_shock']
groups = prediction_clean['patientid']

print(f"\n📋 Dataset for Modeling:")
print(f"   Total samples: {len(y):,}")
print(f"   Positive class: {y.sum():,} ({y.mean():.1%})")
print(f"   Unique patients: {groups.nunique():,}")
print(f"   Avg windows/patient: {len(y) / groups.nunique():.1f}")
print(f"\n📊 Missingness after imputation:")
print(f"   Overall missing rate: {prediction_clean.drop(columns=['patientid', 'target_septic_shock']).isna().mean().mean():.2%}")

In [ ]:
# Repeated Cross-validation using expanded feature sets
n_repeats = 10
n_folds = 5
n_folds_total = n_repeats * n_folds

comparison_results = {}

print(f"\n🎯 Training models with {n_repeats}-Repeat {n_folds}-Fold CV (Expanded Sets):\n")

for feature_set_name, feature_cols in feature_sets.items():
    print(f"{feature_set_name}")
    print("-" * 60)
    fold_metrics = train_repeated_cv(
        prediction_df=prediction_clean,
        feature_cols=feature_cols,
        target_col='target_septic_shock',
        group_col='patientid',
        n_repeats=n_repeats,
        n_folds=n_folds,
        random_state=920
    )
    comparison_results[feature_set_name] = fold_metrics
    print(f"  ROC-AUC:           {np.mean(fold_metrics['roc_auc']):.3f} ± {np.std(fold_metrics['roc_auc']):.3f}")
    print(f"  Average Precision: {np.mean(fold_metrics['avg_precision']):.3f} ± {np.std(fold_metrics['avg_precision']):.3f}\n")

In [ ]:
# Performance comparison (expanded)
summary_df = pd.DataFrame([
    {
        'Model': name,
        'ROC-AUC': f"{np.mean(metrics['roc_auc']):.3f} ± {np.std(metrics['roc_auc']):.3f}",
        'Avg Precision': f"{np.mean(metrics['avg_precision']):.3f} ± {np.std(metrics['avg_precision']):.3f}"
    }
    for name, metrics in comparison_results.items()
])

print("\n📊 Performance Summary (Expanded Sets):")
print(summary_df.to_string(index=False))

fig, (ax1, ax2) = plot_roc_pr_curves(
    comparison_results=comparison_results,
    outcome_df=prediction_clean,
    target_col='target_septic_shock',
    color_scheme='Set1'
)
plt.savefig('../../results/hirid/sepsis/model_comparison_curves_expanded.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: results/hirid/sepsis/model_comparison_curves_expanded.png")
plt.show()

pairs_to_compare = [(0, 1), (1, 2)]
fig, (ax1, ax2) = plot_boxplots_with_stats(
    comparison_results=comparison_results,
    outcome_df=prediction_clean,
    target_col='target_septic_shock',
    pairs_to_compare=pairs_to_compare,
    n_folds_total=n_folds_total
)
plt.savefig('../../results/hirid/sepsis/model_comparison_boxplots_expanded.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: results/hirid/sepsis/model_comparison_boxplots_expanded.png")
plt.show()

print_statistical_comparisons(
    comparison_results=comparison_results,
    pairs_to_compare=pairs_to_compare
)